In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

## Baseline Logistic Regression Model
Model to predict whether a flight will be delayed by more than 15 minutes.
* Start with single predictor (prev_arrival_delay): arrival delay of the previous flight flown by the same aircraft.
  * [actual arrival time - scheduled arrival time].
* Response (delay_indicator): Flight departed more than 15 minutes late.
* Training/testing set split: 80/20.
* Random sample of 500K rows selected to work around session crashes due to high memory usage (happened with full multiple predictor model).
* Specifying seed with "random_state=0" for reproducibility.



In [4]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load cleaned data
df = pd.read_csv('/content/drive/MyDrive/654/project/cleaned_flight_data.csv')

# Drop rows where prev_arrival_delay is missing
df_model = df.dropna(subset=['prev_arrival_delay'])

# Sampling 500K rows to avoid session crash [memory usage] with multiple predictors.
df_model = df_model.sample(n=500000, random_state=0)

# Train/test split (80/20)
train, test = train_test_split(df_model, test_size=0.2, random_state=0)

# Fit logistic regression on training set
model = smf.glm(
    formula='delay_indicator ~ prev_arrival_delay',
    data=train,
    family=sm.families.Binomial()
).fit()

print(model.summary())

# Predict on test set
test = test.copy()
test['pred_prob'] = model.predict(test)
test['pred_class'] = (test['pred_prob'] > 0.5).astype(int)

# Percentage of flights delayed
print(df['delay_indicator'].value_counts(normalize=True))

# Accuracy
accuracy = np.mean(test['pred_class'] == test['delay_indicator'])
print('Accuracy:', accuracy)

# AUC
auc = roc_auc_score(test['delay_indicator'], test['pred_prob'])
print('AUC:', auc)

# Confusion matrix
print(pd.crosstab(test['pred_class'], test['delay_indicator'],
                  rownames=['Predicted'], colnames=['Actual']))

                 Generalized Linear Model Regression Results                  
Dep. Variable:        delay_indicator   No. Observations:               400000
Model:                            GLM   Df Residuals:                   399998
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.5254e+05
Date:                Sun, 19 Apr 2026   Deviance:                   3.0509e+05
Time:                        02:37:26   Pearson chi2:                 2.86e+16
No. Iterations:                     7   Pseudo R-squ. (CS):             0.1574
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -1.7751      0

## Output (Baseline Logistic Regression Model):
Logistic regression model:
* log(p/(1-p)) = -1.7751 + 0.0481 * prev_arrival_delay
* For every additional minute of delay on the previous flight, the log-odds of the current flight being delayed increases by 0.0481; indicating that if the previous flight arrived late, the next flight by that same aircraft is more likely to depart late.
* p-value = 0.000, which means that we can be very confident that previous arrival delay predicts the departure delay.
* AUC of 0.776 indicates the model ranks delayed flights above non-delayed flights well above chance (0.5).
* From the confusion matrix, the model correctly identifies 5,047 delayed flights but misses 12,656, catching only about 28% of actual delayed flights, which means that other factors are also at play.


## Baseline+1 Logistic Regression Model
* Add turnaround_time as a predictor.
  * [actual departure time - actual arrival time of the previous flight].

In [5]:
# Fit logistic regression with two predictors
model2 = smf.glm(
    formula='delay_indicator ~ prev_arrival_delay + turnaround_time',
    data=train,
    family=sm.families.Binomial()
).fit()

print(model2.summary())

# Predict on test set
test['pred_prob2'] = model2.predict(test)
test['pred_class2'] = (test['pred_prob2'] > 0.5).astype(int)

# Accuracy
accuracy2 = np.mean(test['pred_class2'] == test['delay_indicator'])
print('Accuracy:', accuracy2)

# AUC
mask = test['pred_prob2'].notna()
auc2 = roc_auc_score(test.loc[mask, 'delay_indicator'], test.loc[mask, 'pred_prob2'])
print('AUC:', auc2)

# confusion matrix
print(pd.crosstab(test.loc[mask, 'pred_class2'], test.loc[mask, 'delay_indicator'],
                  rownames=['Predicted'], colnames=['Actual']))

                 Generalized Linear Model Regression Results                  
Dep. Variable:        delay_indicator   No. Observations:               397198
Model:                            GLM   Df Residuals:                   397195
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.5110e+05
Date:                Sun, 19 Apr 2026   Deviance:                   3.0219e+05
Time:                        02:37:58   Pearson chi2:                 2.89e+16
No. Iterations:                     7   Pseudo R-squ. (CS):             0.1591
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -1.7831      0

## Output (Baseline+1 Logistic Regression Model):
* Model: log(p/(1-p)) = -1.7831 + 0.0484 * prev_arrival_delay + 0.0000191 * turnaround_time
* Both predictors significant (p=0.000).
* AUC improved slightly (0.776 to 0.785).
* Confusion matrix mostly the same, still catching only about 28% of actual delays.


## Full Logistic Model
* Add all predictors:
  * prev_arrival_delay
  * turnaround_time
  * airport_congestion
  * hour
  * DAY_OF_WEEK
  * AIRLINE

In [6]:
# Fit logistic regression with all predictors
model3 = smf.glm(
    formula='delay_indicator ~ prev_arrival_delay + turnaround_time + airport_congestion + hour + DAY_OF_WEEK + C(AIRLINE)',
    data=train,
    family=sm.families.Binomial()
).fit()

print(model3.summary())

# Predict on test set
test['pred_prob3'] = model3.predict(test)
test['pred_class3'] = (test['pred_prob3'] > 0.5).astype(int)

# Accuracy
mask3 = test['pred_prob3'].notna()
accuracy3 = np.mean(test.loc[mask3, 'pred_class3'] == test.loc[mask3, 'delay_indicator'])
print('Accuracy:', accuracy3)

# AUC
auc3 = roc_auc_score(test.loc[mask3, 'delay_indicator'], test.loc[mask3, 'pred_prob3'])
print('AUC:', auc3)

# Confusion matrix
print(pd.crosstab(test.loc[mask3, 'pred_class3'], test.loc[mask3, 'delay_indicator'],
                  rownames=['Predicted'], colnames=['Actual']))

                 Generalized Linear Model Regression Results                  
Dep. Variable:        delay_indicator   No. Observations:               397198
Model:                            GLM   Df Residuals:                   397179
Model Family:                Binomial   Df Model:                           18
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.4013e+05
Date:                Sun, 19 Apr 2026   Deviance:                   2.8026e+05
Time:                        02:38:26   Pearson chi2:                 1.59e+16
No. Iterations:                     7   Pseudo R-squ. (CS):             0.2043
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -2.8057      0

## Output (Full Logistic Model):
* All 6 predictors significant (p < 0.05).
  * MQ (Envoy Air) and VX (Virgin America) not significant relative to baseline (AA), but AIRLINE retained as a whole.
* AUC improved from 0.785 to 0.830.
* Confusion matrix: catching about 35% of actual delays (6,071 of 17,573), up from 28%.
* UA (United) most delay-prone airline relative to baseline (AA). HA (Hawaiian) least delay-prone.


## Regularization: Lasso Logistic Model

Applied Lasso (L1 regularization) to the logistic regression model to validate predictor selection and check for overfitting.

In [7]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Prepare features and response
features = ['prev_arrival_delay', 'turnaround_time', 'airport_congestion', 'hour', 'DAY_OF_WEEK']

# Get airline dummies
train_dummies = pd.get_dummies(train, columns=['AIRLINE'], drop_first=True)
test_dummies = pd.get_dummies(test, columns=['AIRLINE'], drop_first=True)

# Build feature matrix
airline_cols = [c for c in train_dummies.columns if c.startswith('AIRLINE_') and c != 'AIRLINE_DELAY']
all_features = features + airline_cols

# Drop NaNs
train_clean = train_dummies[all_features + ['delay_indicator']].dropna()
test_clean = test_dummies[all_features + ['delay_indicator']].dropna()

X_train = train_clean[all_features]
y_train = train_clean['delay_indicator']
X_test = test_clean[all_features]
y_test = test_clean['delay_indicator']

# Standardize (required for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit Lasso logistic regression with cross-validation
lasso_model = LogisticRegressionCV(
    penalty='l1',
    solver='saga',
    cv=3,
    random_state=0,
    max_iter=1000
)
lasso_model.fit(X_train_scaled, y_train)

# Results
y_pred = lasso_model.predict(X_test_scaled)
y_prob = lasso_model.predict_proba(X_test_scaled)[:, 1]

print('Best lambda:', 1/lasso_model.C_[0])
print('Coefficients:')
print(pd.Series(lasso_model.coef_[0], index=all_features))
print('Accuracy:', accuracy_score(y_test, y_pred))
print('AUC:', roc_auc_score(y_test, y_prob))
print(pd.crosstab(y_pred, y_test, rownames=['Predicted'], colnames=['Actual']))

Best lambda: 2.782559402207126
Coefficients:
prev_arrival_delay    1.310662
turnaround_time       0.058238
airport_congestion    0.646605
hour                  0.202548
DAY_OF_WEEK          -0.011237
AIRLINE_AS           -0.066493
AIRLINE_B6            0.017114
AIRLINE_DL           -0.036636
AIRLINE_EV           -0.036665
AIRLINE_F9           -0.011389
AIRLINE_HA           -0.078894
AIRLINE_MQ            0.008852
AIRLINE_NK            0.009483
AIRLINE_OO           -0.053862
AIRLINE_UA            0.113464
AIRLINE_US           -0.046752
AIRLINE_VX           -0.004443
AIRLINE_WN            0.072939
dtype: float64
Accuracy: 0.8693231023301714
AUC: 0.8297557808722991
Actual         0      1
Predicted              
0          80260  11504
1           1473   6069


## Output (Lasso Logistic Model):
* Best $\lambda$ = 2.783.
* No coefficients zeroed out.
* DAY_OF_WEEK coefficient near zero (-0.011).
* AUC of 0.830, identical to model without regularization, confirming no overfitting.

## Linear Regression Model
Predicting actual departure delay in minutes (DEPARTURE_DELAY) rather than whether a flight will be delayed. Same predictors as in the full Classification Model.

In [8]:
# Fit linear regression
reg_model = smf.ols(
formula='DEPARTURE_DELAY ~ prev_arrival_delay + turnaround_time + airport_congestion + hour + DAY_OF_WEEK + C(AIRLINE)',
    data=train
).fit()

print(reg_model.summary())

# Predict on test set
test['pred_delay'] = reg_model.predict(test)

# Drop NaNs
mask_reg = test['pred_delay'].notna()

# RMSE
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(test.loc[mask_reg, 'DEPARTURE_DELAY'], test.loc[mask_reg, 'pred_delay']))
print('RMSE:', rmse)

# R-squared
from sklearn.metrics import r2_score
r2 = r2_score(test.loc[mask_reg, 'DEPARTURE_DELAY'], test.loc[mask_reg, 'pred_delay'])
print('R-squared:', r2)

                            OLS Regression Results                            
Dep. Variable:        DEPARTURE_DELAY   R-squared:                       0.319
Model:                            OLS   Adj. R-squared:                  0.319
Method:                 Least Squares   F-statistic:                 1.036e+04
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        02:41:48   Log-Likelihood:            -1.8962e+06
No. Observations:              397198   AIC:                         3.792e+06
Df Residuals:                  397179   BIC:                         3.793e+06
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              1.3592      0

## Output:

* R-squared of 0.319, model explains about 32% of variation in departure delay minutes. Remaining 68% reflects factors excluded or not captured in the dataset (weather, mechanical issues, air traffic control etc.).
* RMSE of 28.15 minutes, predictions are off by about 28 minutes on average.
* prev_arrival_delay coefficient = 0.466: for every extra minute the previous flight was late, the current departure is expected to be about 0.47 minutes later.
  * This is the strongest predictor.
* airport_congestion coefficient = 0.779, more congested airports translate directly to longer delays.
* DAY_OF_WEEK significant (p=0.000), negative coefficient (-0.097).
  * Negative coefficient suggests "later days" of the week have shorter delays.
  * (What is the week-day encoding)?
* hour significant (p=0.004), negative coefficient (-0.030), indicating later departures associated with slightly shorter delays.
  * inconsistent with positive coefficient in Classification model (0.0409)?
* UA (United) most delay-prone (+4.39 minutes vs baseline AA). HA (Hawaiian) and AS (Alaska) least delay-prone.
  * Several airlines not significant: B6, MQ, NK, VX.

## Regularization: Lasso Linear Model
Applied Lasso (L1 regularization) to the linear regression model to validate predictor selection and check for overfitting.

In [9]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

# Prepare features
features_reg = ['prev_arrival_delay', 'turnaround_time', 'airport_congestion', 'hour', 'DAY_OF_WEEK']
airline_cols_reg = [c for c in train_dummies.columns if c.startswith('AIRLINE_') and c != 'AIRLINE_DELAY']
all_features_reg = features_reg + airline_cols_reg

# Drop NaNs
train_reg = train_dummies[all_features_reg + ['DEPARTURE_DELAY']].dropna()
test_reg = test_dummies[all_features_reg + ['DEPARTURE_DELAY']].dropna()

X_train_reg = train_reg[all_features_reg]
y_train_reg = train_reg['DEPARTURE_DELAY']
X_test_reg = test_reg[all_features_reg]
y_test_reg = test_reg['DEPARTURE_DELAY']

# Standardize
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

# Fit Lasso regression with cross-validation
lasso_reg = LassoCV(cv=3, random_state=0, max_iter=5000)
lasso_reg.fit(X_train_reg_scaled, y_train_reg)

# Results
y_pred_reg = lasso_reg.predict(X_test_reg_scaled)
print('Best lambda:', lasso_reg.alpha_)
print('Coefficients:')
print(pd.Series(lasso_reg.coef_, index=all_features_reg))

rmse_lasso = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2_lasso = r2_score(y_test_reg, y_pred_reg)
print('RMSE:', rmse_lasso)
print('R-squared:', r2_lasso)

Best lambda: 0.04321329234670556
Coefficients:
prev_arrival_delay    13.890777
turnaround_time        0.862091
airport_congestion    11.020062
hour                  -0.071691
DAY_OF_WEEK           -0.152612
AIRLINE_AS            -0.386561
AIRLINE_B6            -0.000000
AIRLINE_DL             0.320851
AIRLINE_EV            -0.183990
AIRLINE_F9            -0.037497
AIRLINE_HA            -0.215287
AIRLINE_MQ            -0.000000
AIRLINE_NK            -0.016706
AIRLINE_OO            -0.310202
AIRLINE_UA             1.270196
AIRLINE_US            -0.239399
AIRLINE_VX            -0.000000
AIRLINE_WN             0.517169
dtype: float64
RMSE: 28.146835292995554
R-squared: 0.3166764973718451


## Output:
* Best lambda = 0.043.
* Lasso zeroed out B6, MQ, and VX coefficients.
  * Same airlines were not significant in the linear regression model.
* RMSE of 28.15 and R-squared of 0.317, identical to linear regression, confirming no overfitting.
* Lasso validated the linear regression model, and confirmed B6, MQ, and VX do not contribute to predicting delay minutes.

## Model Comparison
Summary of all models with key performance metrics.

In [10]:
results = pd.DataFrame({
    'Model': ['Baseline Logistic', 'Baseline+1 Logistic', 'Full Logistic', 'Lasso Logistic', 'Linear Regression', 'Lasso Linear'],
    'Type': ['Classification', 'Classification', 'Classification', 'Classification', 'Regression', 'Regression'],
    'AUC': [0.7767, 0.7854, 0.8284, 0.8284, None, None],
    'Accuracy': [0.8629, 0.8630, 0.8691, 0.8691, None, None],
    'RMSE': [None, None, None, None, 27.87, 27.87],
    'R_squared': [None, None, None, None, 0.324, 0.324]
})

print(results.to_string(index=False))

              Model           Type    AUC  Accuracy  RMSE  R_squared
  Baseline Logistic Classification 0.7767    0.8629   NaN        NaN
Baseline+1 Logistic Classification 0.7854    0.8630   NaN        NaN
      Full Logistic Classification 0.8284    0.8691   NaN        NaN
     Lasso Logistic Classification 0.8284    0.8691   NaN        NaN
  Linear Regression     Regression    NaN       NaN 27.87      0.324
       Lasso Linear     Regression    NaN       NaN 27.87      0.324
